# Dijet systematic uncertainties: old vs new

Compare the relative total uncertainties on the unit-normalized dijet shape and the forward/backward yield ratio. Old refers to the shaded systematic bands in [CMS-PAS-HIN-24-014, Figs. 1–4](https://cds.cern.ch/record/2917837/files/HIN-24-014-pas.pdf). The archive also contains a separate local black Total curve for full η, with different values, some clipped steps and different high-$p_{\mathrm{T}}^{\mathrm{ave}}$ bins. It is not substituted for the measurement bands.

## From laboratory to CM pseudorapidity

1. **Orient the laboratory axis.** [CMS-PAS-HIN-24-014, Sec. 2](https://cds.cern.ch/record/2917837/files/HIN-24-014-pas.pdf) defines positive η toward the proton beam after aligning the two beam configurations. This is already the convention of the old full-η ROOT graphs. JetAnalysis starts from raw forest η and implements the alignment in `AnalysisContract.h`:

   | Forest input | Pb-going flag set | Pb-going flag unset |
   |---|---:|---:|
   | Data | $\eta_{\rm lab}=+\eta_{\rm raw}$ | $\eta_{\rm lab}=-\eta_{\rm raw}$ |
   | MC | $\eta_{\rm lab}=-\eta_{\rm raw}$ | $\eta_{\rm lab}=+\eta_{\rm raw}$ |

   These signs apply to raw forest coordinates only. Applying a reflection to the old graphs would reverse an axis that was already aligned.
2. **Subtract the CM offset.** A massless particle at CM η = 0 appears at laboratory η = +0.465 along the proton-positive axis. The project therefore uses $\eta_{\rm CM}^{\rm jet}=\eta_{\rm lab}^{\rm jet}-0.465$. In `processForestSimple.C`, the dijet coordinate is the mean of the two shifted jet values, so
   $$
   \eta_{\rm CM}^{\rm dijet}
   = \tfrac12[(\eta_{1,\rm lab}-0.465)+(\eta_{2,\rm lab}-0.465)]
   = \eta_{\rm lab}^{\rm dijet}-0.465.
   $$
3. **Translate each archived full-η bin.** For graph center $x$ and horizontal errors $e_-,e_+$, the transformed center is $x-0.465$ and the transformed edges are $x-e_--0.465$ and $x+e_+-0.465$. For example, the old bin $[0.4,0.6]$ centered at 0.5 becomes $[-0.065,0.135]$ centered at 0.035. The width and relative systematic uncertainty are unchanged. The optional marker offset is applied only when drawing coincident full-η centers; it never enters the bin matching or the numerical output.

The translation implements the project's fixed-shift pseudorapidity convention, which approximates the rapidity transformation for massless jets. It does not boost jet four-vectors or reapply event selection. The old F/B graphs already use $|\eta_{\rm CM}^{\rm dijet}|$, with positive CM η in the numerator and reflected negative CM η in the denominator. Neither the old F/B nor any of the new coordinates are translated.

The PAS uses per-jet $|\eta_{\mathrm{lab}}|<3$ for the full distribution and $|\eta_{\mathrm{CM}}|<2.5$ for F/B, and $|\Delta\phi|>5\pi/6$. The current run uses Jet ID, $|\eta_{\mathrm{CM}}^{\mathrm{jet}}|\leq1.9$, $|\Delta\phi|\geq2\pi/3$, and 50/40 GeV thresholds. The historical variations also differ from the current four-source total. The comparison measures differences in uncertainty size; it cannot isolate their cause or establish statistical significance.

The old values are PDF vector extractions. The stored 0.003 PDF-point tolerance describes extraction precision; it is not an additional systematic error. No uncertainty is added for digitization. New values come from a pinned archive of `04_systematics_total.ipynb`, with JER, JEU, pileup and pointing resolution combined in quadrature. To refresh, run that producer notebook with the desired declared inputs and point this notebook at the resulting ROOT file.

In [ ]:
import os
from pathlib import Path
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "AnalysisContract.h").is_file() and (p / "hist_analysis").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from the jetAnalysis repository.")
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from IPython.display import Image, display

from hist_analysis.config.histograms import DIJET_PTAVE_BINS as dijet_ptave_bins
from hist_analysis.python.notebook_setup import load_root
from hist_analysis.python.output_naming import OutputDirectory, OutputSelection
from hist_analysis.python.plot_labels import PPB_MPLHEP_HEADER_RIGHT
from hist_analysis.python.systematics_comparison import (
    compare_bins, plot_comparisons, read_current, read_published, save_mplhep_comparison,
)

In [ ]:
# Inputs: explicit archive so a different run is never selected silently.
OLD_ROOT_FILE = PROJECT_ROOT / "hist_analysis/root_files/old_distributions/pPb8160_dijet_published_distributions.root"
NEW_SYSTEMATICS_DIR = PROJECT_ROOT / "hist_analysis/output/systematics_total"
NEW_ROOT_FILE = (NEW_SYSTEMATICS_DIR / "2026-11-12/jetId/root"
                 / "jetId_etaCM1p9_ptAveBins5ce9da0b640f9819_jetId_normal_04_systematics_total.root")
OUTPUT_ROOT = Path(os.environ.get(
    "DIJET_OLD_VS_NEW_SYSTEMATICS_OUTPUT_DIR",
    PROJECT_ROOT / "hist_analysis/output/systematics_old_vs_new",
))

PTAVE_BINS = dijet_ptave_bins
SELECTED_PTAVE_BINS = None  # Optional subset of PTAVE_BINS; keeps canonical order.
PLOT_FRAME = "cm"
LAB_TO_CM_SHIFT = -0.465
EXPECTED_NEW = dict(SELECTION="jetId", ETA_CUT=1.9, DIRECTION="combined",
                    DIJET_BINNING="normal", OBSERVABLE="both",
                    POINTING_REFERENCE_DEFINITION="refWithRecoPtAve")

OLD_STYLE = {"color": "black", "marker": "o"}
NEW_STYLE = {"color": "tab:red", "marker": "o", "markerfacecolor": "none"}
FULL_ETA_YLIM = (0., 20.)  # Percent. Use None to scale automatically.
FB_YLIM = (0., 10.)        # Percent. Use None to scale automatically.
FULL_ETA_MARKER_OFFSET = 0.015  # 0 disables; only applied to coincident centers.
CMS_HEADER_LEFT = "CMS Preliminary"
CMS_HEADER_RIGHT = PPB_MPLHEP_HEADER_RIGHT
SAVE_MPLHEP = True      # Write figures to the dated mplhep directory.
SAVE_MPLHEP_PNG = False  # Also save PNG when included in OUTPUT_FORMATS.
SAVE_TABLES = True
OUTPUT_FORMATS = ["pdf"] # "pdf", "png"
SHOW_INDIVIDUAL = False  # Save all overlays; display grid previews by default.

In [ ]:
if PLOT_FRAME != "cm":
    raise ValueError("Only a common CM comparison is supported.")

ROOT = load_root()
with ROOT.TFile.Open(str(OLD_ROOT_FILE), "READ") as archive:
    old, old_provenance = read_published(archive, LAB_TO_CM_SHIFT)
with ROOT.TFile.Open(str(NEW_ROOT_FILE), "READ") as archive:
    new, new_configuration = read_current(archive, EXPECTED_NEW)

requested = set(PTAVE_BINS if SELECTED_PTAVE_BINS is None else map(tuple, SELECTED_PTAVE_BINS))
if requested - set(PTAVE_BINS):
    raise ValueError(f"Noncanonical pTave bins: {requested - set(PTAVE_BINS)}")

panel_sets = {}
for observable in ("full", "fb"):
    missing_old = [pt for pt in PTAVE_BINS if pt in requested and (observable, pt) not in old]
    print(f"{observable}: no PAS measurement band for {missing_old or 'none'}")
    intervals = [pt for pt in PTAVE_BINS if pt in requested and (observable, pt) in old]
    if not intervals:
        raise ValueError(f"No comparable {observable} intervals")
    panel_sets[observable] = [(pt, old[observable, pt], new[observable, pt]) for pt in intervals]

# Fixed F/B domain from the full historical measurement, independent of subset.
fb_max = max(table.high.max() for (observable, pt), table in old.items() if observable == "fb")
print(f"F/B axis: [0, {fb_max:g}]; old extraction: {old_provenance['digitization']}")

## Numerical comparison

The new histograms extend to ±2.0 (full) or 2.0 (F/B) because their outer bins straddle the 1.9 fiducial boundary. Matching nominal edges does not imply identical effective acceptance, especially for the outer F/B bin.

Require both physical edges to agree (absolute tolerance (10^{-9}) in η for rounding). Unmatched bins still appear in the overlays. Differences are New − Old in percentage points; the per-bin table preserves the η dependence. Zero matches mean no defensible binwise differences can be reported.

In [ ]:
coverage_rows, matched_rows = [], []
for observable, panels in panel_sets.items():
    for (low, high), previous, current in panels:
        matched = compare_bins(previous, current)
        coverage_rows.append(dict(observable=observable, pt_low=low, pt_high=high,
                                  old_bins=len(previous), new_bins=len(current), matched=len(matched),
                                  mean_delta_pp=matched.delta_pp.mean(),
                                  mean_abs_delta_pp=matched.abs_delta_pp.mean(),
                                  max_abs_delta_pp=matched.abs_delta_pp.max()))
        matched_rows.append(matched.assign(observable=observable, pt_low=low, pt_high=high))

coverage = pd.DataFrame(coverage_rows)
differences = pd.concat(matched_rows, ignore_index=True)
display(coverage.round(3))
for observable in panel_sets:
    rows = differences[differences.observable == observable]
    if rows.empty:
        print(f"{observable}: no edge-matched bins; no signed trend inferred.")
        continue
    print(f"{observable}: {len(rows)} matches; mean signed Δ = {rows.delta_pp.mean():.3f} pp; "
          f"mean |Δ| = {rows.abs_delta_pp.mean():.3f} pp; max |Δ| = {rows.abs_delta_pp.max():.3f} pp; "
          f"new lower in {(rows.delta_pp < 0).sum()} bins and higher in {(rows.delta_pp > 0).sum()} bins.")
    display(rows.nlargest(5, "abs_delta_pp").round(3))
    display(rows.groupby(["low", "high"]).delta_pp.agg(["count", "mean", "min", "max"]).round(3))

In [ ]:
from io import BytesIO
from matplotlib import pyplot as plt

selection = new_configuration["SELECTION"]
eta_cut = new_configuration["ETA_CUT"]
cuts = (
    (rf"$|\eta_{{\mathrm{{CM}}}}^{{\mathrm{{jet}}}}|\leq {eta_cut:g}$",
     r"$|\Delta\phi|\geq 2\pi/3$"),
    (r"$p_{\mathrm{T}}^{\mathrm{lead}}\geq 50$ GeV",
     r"$p_{\mathrm{T}}^{\mathrm{sublead}}\geq 40$ GeV"),
)
save_figures = SAVE_MPLHEP and ("pdf" in OUTPUT_FORMATS
                                 or (SAVE_MPLHEP_PNG and "png" in OUTPUT_FORMATS))
compared_intervals = {pt for panels in panel_sets.values() for pt, _, _ in panels}
if SAVE_TABLES or save_figures:
    output_selection = OutputSelection(
        selection, eta_cut, [pt for pt in PTAVE_BINS if pt in compared_intervals], "ptAve",
    )
    OUTPUT_DIR = OutputDirectory(OUTPUT_ROOT, output_selection, workflow_root=OUTPUT_ROOT)
    print(f"Comparison output: {OUTPUT_DIR.path.parent}")
if SAVE_TABLES:
    coverage.to_csv(OUTPUT_DIR / "comparison_coverage.csv", index=False)
    differences.to_csv(OUTPUT_DIR / "matched_bin_differences_pp.csv", index=False)

def render(observable, panels, *, grid, xlim, ylim):
    fig = plot_comparisons(
        observable, panels, grid=grid, xlim=xlim, ylim=ylim,
        styles=(OLD_STYLE, NEW_STYLE), marker_offset=FULL_ETA_MARKER_OFFSET,
        cuts=cuts, header=(CMS_HEADER_LEFT, CMS_HEADER_RIGHT),
    )
    if save_figures:
        identity = OutputSelection(selection, eta_cut, [panel[0] for panel in panels], "ptAve")
        stem = f"{observable}_old_vs_new_{'grid4x4' if grid else 'individual'}"
        figure_dir = OUTPUT_DIR / "mplhep" / observable
        extension = "pdf" if "pdf" in OUTPUT_FORMATS else "png"
        path = figure_dir.file(f"{stem}.{extension}", selection=identity)
        if extension == "pdf" and SAVE_MPLHEP_PNG and "png" in OUTPUT_FORMATS:
            figure_dir.file(f"{stem}.png", selection=identity)
        save_mplhep_comparison(
            fig, path.with_suffix(".pdf"), grid=grid,
            save_png=SAVE_MPLHEP_PNG, formats=OUTPUT_FORMATS,
            partial_grid=grid and len(panels) < 16,
        )
    if grid or SHOW_INDIVIDUAL:
        preview = BytesIO()
        fig.savefig(preview, format="png", dpi=85)
        display(Image(data=preview.getvalue()))
    plt.close(fig)

for observable, panels in panel_sets.items():
    curves = [table for _, old_table, new_table in panels for table in (old_table, new_table)]
    xlim = ((0, fb_max) if observable == "fb" else
            (min(t.low.min() for t in curves), max(t.high.max() for t in curves)))
    outside = [(pt, int((~table.eta.between(*xlim)).sum()))
               for pt, _, table in panels if not table.eta.between(*xlim).all()]
    if outside:
        print(f"{observable}: new centers outside the display domain (interval, count): {outside}")
    visible = pd.concat(curves).loc[lambda t: t.eta.between(*xlim), "percent"]
    limits = FB_YLIM if observable == "fb" else FULL_ETA_YLIM
    ylim = (0, max(1., 1.5 * visible.max())) if limits is None else limits
    if not np.isfinite(ylim).all() or not ylim[0] < ylim[1]:
        raise ValueError(f"{observable}: y limits must be finite and increasing")
    clipped = (~visible.between(*ylim)).sum()
    if clipped:
        print(f"{observable}: {clipped} points fall outside the configured y range {ylim}")
    for panel in panels:
        local = pd.concat(panel[1:]).loc[lambda t: t.eta.between(*xlim), "percent"]
        local_ylim = (0, max(1., 1.3 * local.max())) if limits is None else ylim
        render(observable, [panel], grid=False, xlim=xlim, ylim=local_ylim)
    for start in range(0, len(panels), 16):
        render(observable, panels[start:start + 16], grid=True, xlim=xlim, ylim=ylim)
